# 00 · The whole pipeline in one process

**Goal:** watch every object the two sides exchange, in the order they appear, without any cloud service.

| Notebook | Topic | Pitfalls it covers |
| --- | --- | --- |
| 00 | the whole flow, one step at a time | who writes which file |
| 01 | Left sources: Oracle, files, credentials | secrets vs `conn_macro`, raw text |
| 02 | the store protocol on S3 | pointer written last, zero padding, folder markers |
| 03 | row check, gate, manifest | `manual` is terminal, the verdict is the commit marker |
| 04 | Databricks reads | `DESCRIBE QUERY`, deterministic sampling, type mismatch |
| 05 | col check statistics | decimal collapsing, unit counting, WIP, scheduling |
| 06 | job and report | what makes a run `PARTIAL` |
| 07 | the `dtrack` CLI | stdout/stderr, exit codes, run id = folder |
| 08 | Databricks SDK vs the UI | `jobs.reset` vs `update`, managed storage, grants |
| 09 | manual handoff from SAS / Excel / Athena extracts | extract conventions, names and types via `col_map` |
| 10 | preload Left statistics before the Right side is ready | cutoff, top-up, stale detection, forced re-upload |
| 11 | a run that adopts the preload and computes the rest | what invalidates a preloaded unit |
| 12 | developer workflow: code map, test tiers, enforced rules | change → tests table, definition of done |
| 13 | change recipes: add, remove, modify, fix | blast radius with `impact()`, regression tests for real bugs |

Here the "S3 bucket" is a local folder (`VolumeStore`, the same store contract as `S3Store`),
the "Oracle" is a set of CSV files read through DuckDB, and the "Databricks table" is a CSV
read by `dtrack_local.csv_target.CsvAnchorTarget`. Every reconciliation function called below is production code.

```text
LEFT (this PC)                      run store (S3 = Volume)                 RIGHT (Databricks job)
row-check  ── row_check/<pair>.left.json ──►                     ──► reconcile, gate
           ── row_check/_LEFT_CLOSED     ──►     reports/<pair>_compare_row.csv  ◄──
                                                 manifest/<pair>.json            ◄── (gate passed)
                                                 row_check/<pair>.json           ◄── (verdict, last)
                                                 row_check/_CLOSED               ◄── (all pairs done)
col-check  ◄── wait for row_check/_CLOSED, read manifest/
           ── units/<id>.json, then _INDEX/<seq>-<id>.json ──►  ──► compare
           ── manifest/_CLOSED {expected_units} ──►              results/<id>.json ◄──
fetch-report ◄── history/<run>/…, latest/pointer.json ◄────────── report
```

In [ ]:
import json

import nbkit
import pandas as pd
from nbkit import code, show, table, tree

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

In [ ]:
folder = nbkit.sandbox("00")
data, config, summary = nbkit.dataset(folder)
print(
    sorted(p.name for p in (data / "left").iterdir()),
    sorted(p.name for p in (data / "right").iterdir()),
)
show(summary["orders"], "injected differences (orders)")

## Step 1 · Left row check

`run_row_check` counts rows per partition on the Left source and writes one document per pair.
The config it writes has no credentials (see notebook 01). `_LEFT_CLOSED` tells the Right side
which pairs to expect.

In [ ]:
from dtrack_left.checks.row_check import run_row_check as left_row_check
from dtrack_right.backends.store import VolumeStore

store = VolumeStore(folder / "runs" / "nb00")
run_id, documents = left_row_check(config, store, run_id="nb00")
tree(store.root)
show(store.get_json("row_check/_LEFT_CLOSED"), "row_check/_LEFT_CLOSED")

## Step 2 · Right row check

The Right job counts its own side, reconciles every partition, applies the gate, and writes
the CSV report and the manifest before the verdict document (the verdict is the commit marker,
notebook 03).

In [ ]:
from dtrack_right.checks.row_check import run_row_check as right_row_check

print("pairs reconciled:", right_row_check(store, nbkit.target_factory(data, folder)))
show(store.get_json("row_check/orders.json")["summary"], "row_check/orders.json: summary")
tree(store.root, "row_check/")
tree(store.root, "manifest/")

In [ ]:
manifest = store.get_json("manifest/orders.json")
table(
    pd.DataFrame(
        [
            {"bucket": w["bucket"], "partitions": len(w["partitions"]), "n_left": w["n_left"]}
            for w in manifest["windows"]
        ]
    ).assign(columns=len(manifest["columns"]))
)

## Step 3 · Left col check

The Left side waits for `row_check/_CLOSED`, reads the manifests, pulls each window from the
source into its work folder (the "this folder" of the pipeline), computes statistics and
commits one **unit** per pair × window × column.

In [ ]:
from dtrack_left.checks.col_check import (
    ColCheckPipeline,
    LeftRuntime,
    publish_fixed_keys,
    wait_for_manifests,
)

manifests = wait_for_manifests(store, timeout=1, poll_interval=0)
for item in manifests:
    publish_fixed_keys(store, item, folder / "left_work")
runtime = LeftRuntime(folder / "left_work", store)
published = ColCheckPipeline(
    store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats
).run(manifests)
print("units published:", published)
tree(folder / "left_work")

In [ ]:
tree(store.root, "_INDEX/")
show(store.get_json("manifest/_CLOSED"), "manifest/_CLOSED")

## Step 4 · Right comparison

`ColCheck` reads `_INDEX/` incrementally, computes the Right statistics for each window once,
and compares every column. One `results/<unit>.json` per unit.

In [ ]:
from dtrack_right.checks.col_check import ColCheck

results = ColCheck(store, nbkit.target_factory(data, folder), poll_interval=0).run()
frame = pd.DataFrame(results)[
    [
        "pair",
        "vintage",
        "column",
        "col_type",
        "verdict",
        "hash_sum_match",
        "mean_match",
        "min_match",
        "max_match",
    ]
]
table(frame.sort_values(["pair", "vintage", "column"]))

## Step 5 · Report

`build_report` gathers `row_check/`, `results/`, `dead/` and `manifest/_CLOSED` into
`summary.json`, per-pair CSVs, `index.html` and `report.xlsx`.

In [ ]:
import re

from dtrack_right.report import build_report
from dtrack_right.report.html import summary_section

files = build_report(store, folder / "report")
report = json.loads(files["summary.json"].read_text())
table(pd.DataFrame({"report file": sorted(files)}))
code(
    " ".join(re.sub("<[^>]+>", " ", summary_section(report)).split()),
    label="summary line in index.html",
)

## Who writes what

| Object | Writer | Reader | Meaning |
| --- | --- | --- | --- |
| `row_check/<pair>.left.json` | Left `row-check` | Right job | left counts + public pair config |
| `row_check/_LEFT_CLOSED` | Left `row-check` | Right job | every pair the Left side will send |
| `reports/<pair>_compare_row.csv` | Right job | report | flat verdict per partition |
| `manifest/<pair>.json` | Right job | Left `col-check` | matched partitions grouped into windows |
| `row_check/<pair>.json` | Right job (last) | Right job on restart | verdict; its presence means "pair done" |
| `row_check/_CLOSED` | Right job | Left `col-check` | Stage 1 finished for every pair |
| `units/<id>.json` → `_INDEX/<seq>-<id>.json` | Left `col-check` | Right job | payload first, pointer last |
| `manifest/_CLOSED` | Left `col-check` | Right job, report | Left finished; `expected_units` |
| `results/<id>.json` / `dead/<id>.json` | Right job | report | one comparison or one failure per unit |